# 05 — NSE, screening, and weak-rate tables


This notebook introduces three advanced layers:

1. NSE/equilibrium composition solving.
2. Electron-screening correction factors.
3. Weak-rate interpolation tables and weak source terms.

These are pure-Python replacements intended for transparent workflows and validation against reference data.

In [ ]:
from pathlib import Path
import sys

# Make the local editable package importable when the notebooks are run from the repo.
HERE = Path.cwd()
CANDIDATES = [HERE / "src", HERE.parent / "src", HERE.parent.parent / "src"]
for p in CANDIDATES:
    if (p / "nucnetpy").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nucnetpy as nn
print("nucnetpy version:", nn.__version__)
def build_toy_alpha_network():
    """Build a small alpha-chain toy network for tutorials.

    This is not intended to be a physical helium-burning network.  The rates are
    deliberately simple constants so that the examples run quickly and are easy
    to inspect.
    """
    net = nn.Network()
    for name in ["he4", "be8", "c12"]:
        net.add_species(nn.Species.parse(name))

    r1 = nn.Reaction.from_names(
        ["he4", "he4"], ["be8"],
        constant_rate=2.0e-6,
        q_value=0.092,
        label="toy_2alpha_to_be8",
    )
    r2 = nn.Reaction.from_names(
        ["be8", "he4"], ["c12"],
        constant_rate=5.0e-5,
        q_value=7.367,
        label="toy_be8_alpha_to_c12",
    )
    net.reactions.add(r1)
    net.reactions.add(r2)

    zone = nn.Zone(label=("toy", "0", "0"), properties={"t9": "1.0", "rho": "1e4"})
    zone.set_abundance("he4", 0.25)  # X(he4) = A*Y = 1.0
    zone.set_abundance("be8", 0.0)
    zone.set_abundance("c12", 0.0)
    net.add_zone(zone)
    return net
net = build_toy_alpha_network()
zone = net.zone(0)

## NSE solve

The NSE solver uses species in a network and solves for chemical potentials satisfying `sum(A_i Y_i)=1` and `sum(Z_i Y_i)=Ye`.

For a clean tutorial demonstration we use a one-species helium network with an artificial mass excess chosen so the normalized solution is simple. In production, use real mass excesses and partition functions from your nuclear data file.

In [ ]:
nse_net = nn.Network()
nse_net.add_species(nn.Species('he4', z=2, a=4, mass_excess=-34.0))

nse = nn.solve_nse(nse_net, t9=5.0, rho=1e7, ye=0.5)
print('success:', nse.success, nse.message)
print('mu_p, mu_n:', nse.mu_p, nse.mu_n)
print('xsum:', nse.xsum, 'computed Ye:', nse.computed_ye)
pd.DataFrame([{'species':k, 'Y_NSE':v, 'X_NSE':nn.Species.parse(k).a*v} for k,v in nse.abundances.items()])

## Screening callback

The solver accepts a `screening(reaction, T9, rho, Ye)` callback. The helper below applies the built-in weak-screening model using the current zone composition.

In [ ]:
def screening_callback(reaction, t9, rho, ye):
    context = nn.ScreeningContext(t9=t9, rho=rho, ye=ye, abundances=zone.abundances, species_map=net.species)
    return nn.reaction_screening_factor(reaction, context, model='weak')

for r in net.reactions.reactions:
    print(r.string, screening_callback(r, 1.0, 1e4, zone.ye(net.species)))

## Weak-rate table interpolation

The example table is a small artificial `T9`--`rhoYe` grid.

In [ ]:
weak_path = Path('data') / 'toy_weak_rates.txt'
wr = nn.read_weak_table(weak_path)
print(wr)
print('rate(T9=1.2, rhoYe=2e5) =', wr.rate(1.2, 2e5))
print('nu_loss(T9=1.2, rhoYe=2e5) =', wr.neutrino_loss(1.2, 2e5))

In [ ]:
T9_grid = np.linspace(0.5, 2.0, 50)
rates = [wr.rate(t, 1e5) for t in T9_grid]
plt.plot(T9_grid, rates)
plt.xlabel('T9')
plt.ylabel('weak rate at rhoYe=1e5')
plt.yscale('log');